### TDSP · Fase 2: Adquisición de los datos

Carga de la base de datos, verificación de su estructura y división en grupos de entrenamiento y prueba.

- **Entrada:** `Sample_Data/Raw/online_shoppers_intention.csv`
- **Salidas:**
  - `Sample_Data/Processed/train.csv`, `Sample_Data/Processed/test.csv`: grupos de entrenamiento y prueba
  - `Sample_Data/Processed/split_summary.csv`: resumen de la división
- **Variable objetivo:** `Revenue`

#### Librerías

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

#### Semilla

In [2]:
# Semilla para reproducibilidad. Puede modificarse o eliminarse.
SEED = 1
np.random.seed(SEED)

#### Rutas

Relativas a la carpeta del notebook.

In [3]:
RAW_PATH = Path("../../Sample_Data/Raw/online_shoppers_intention.csv")
PROCESSED_DIR = Path("../../Sample_Data/Processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

#### Carga de datos

In [4]:
df = pd.read_csv(RAW_PATH)

print(f"Registros: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.head()

Registros: 12,330 | Columnas: 18


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


#### Verificación de la estructura

Compara las columnas con las 18 de la ficha de UCI. La celda falla si alguna no coincide.

In [5]:
EXPECTED_COLUMNS = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues", "SpecialDay",
    "Month", "OperatingSystems", "Browser", "Region", "TrafficType",
    "VisitorType", "Weekend", "Revenue",
]

missing_cols = set(EXPECTED_COLUMNS) - set(df.columns)
extra_cols = set(df.columns) - set(EXPECTED_COLUMNS)
assert not missing_cols and not extra_cols, f"Faltan: {missing_cols} | Sobran: {extra_cols}"

print("Las 18 columnas coinciden con la ficha de UCI.")

Las 18 columnas coinciden con la ficha de UCI.


#### Tipos de datos y valores faltantes

In [6]:
pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "faltantes": df.isna().sum(),
    "valores_distintos": df.nunique(),
})

,tipo,faltantes,valores_distintos
Administrative,int64,0,27
Administrative_Duration,float64,0,3335
Informational,int64,0,17
Informational_Duration,float64,0,1258
ProductRelated,int64,0,311
ProductRelated_Duration,float64,0,9551
BounceRates,float64,0,1872
ExitRates,float64,0,4777
PageValues,float64,0,2704
SpecialDay,float64,0,6


#### Verificación de rangos

Conteos y duraciones no negativos; tasas y `SpecialDay` entre 0 y 1.

In [7]:
COUNTS_DURATIONS = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration", "PageValues",
]
RATES = ["BounceRates", "ExitRates", "SpecialDay"]

checks = pd.DataFrame({
    "minimo": df[COUNTS_DURATIONS + RATES].min(),
    "maximo": df[COUNTS_DURATIONS + RATES].max(),
})
checks["fuera_de_rango"] = [
    int((df[c] < 0).sum()) if c in COUNTS_DURATIONS else int(((df[c] < 0) | (df[c] > 1)).sum())
    for c in checks.index
]
checks

,minimo,maximo,fuera_de_rango
Administrative,0.0,27.000000,0
Administrative_Duration,0.0,3398.750000,0
Informational,0.0,24.000000,0
Informational_Duration,0.0,2549.375000,0
ProductRelated,0.0,705.000000,0
ProductRelated_Duration,0.0,63973.522230,0
PageValues,0.0,361.763742,0
BounceRates,0.0,0.200000,0
ExitRates,0.0,0.200000,0
SpecialDay,0.0,1.000000,0


#### Valores de `Month` y `VisitorType`

In [8]:
for col in ["Month", "VisitorType"]:
    print(f"--- {col} ---")
    print(df[col].value_counts().to_string(), "\n")

--- Month ---
Month
May     3364
Nov     2998
Mar     1907
Dec     1727
Oct      549
Sep      448
Aug      433
Jul      432
June     288
Feb      184 

--- VisitorType ---
VisitorType
Returning_Visitor    10551
New_Visitor           1694
Other                   85 



#### Filas repetidas

Filas idénticas en todas las columnas y resumen de su contenido.

In [9]:
dup_mask = df.duplicated(keep=False)

print(f"Filas repetidas (copias adicionales): {df.duplicated().sum()}")
print(f"Filas involucradas en alguna repetición: {dup_mask.sum()}")
print(f"De ellas terminaron en compra: {df.loc[dup_mask, 'Revenue'].sum()}\n")

df.loc[dup_mask, [
    "Administrative", "Informational", "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues",
]].describe().T[["mean", "min", "max"]]

Filas repetidas (copias adicionales): 125
Filas involucradas en alguna repetición: 201
De ellas terminaron en compra: 0



,mean,min,max
Administrative,0.000000,0.0,0.0
Informational,0.000000,0.0,0.0
ProductRelated,1.059701,1.0,2.0
ProductRelated_Duration,0.000000,0.0,0.0
BounceRates,0.200000,0.2,0.2
ExitRates,0.200000,0.2,0.2
PageValues,0.000000,0.0,0.0


#### Distribución de la variable objetivo

In [10]:
pd.DataFrame({
    "sesiones": df["Revenue"].value_counts(),
    "proporcion": df["Revenue"].value_counts(normalize=True).round(4),
})

,sesiones,proporcion
Revenue,,
False,10422,0.8453
True,1908,0.1547


#### Identificador de registro

`row_id` guarda la posición original de cada fila. No se usa como variable predictora.

In [11]:
df = df.reset_index().rename(columns={"index": "row_id"})
df.head(3)

,row_id,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,1,0,0.0,0,0.0,2,64.0,0.0,0.1,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,2,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False


#### División 80/20 estratificada por `Revenue`

In [12]:
TEST_SIZE = 0.20

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    stratify=df["Revenue"],
    random_state=SEED,
)

print(f"Entrenamiento: {len(train_df):,} sesiones")
print(f"Prueba:        {len(test_df):,} sesiones")

Entrenamiento: 9,864 sesiones
Prueba:        2,466 sesiones


#### Verificación de la estratificación

In [13]:
summary = pd.DataFrame({
    "total_n": df["Revenue"].value_counts(),
    "train_n": train_df["Revenue"].value_counts(),
    "test_n": test_df["Revenue"].value_counts(),
    "total_prop": df["Revenue"].value_counts(normalize=True),
    "train_prop": train_df["Revenue"].value_counts(normalize=True),
    "test_prop": test_df["Revenue"].value_counts(normalize=True),
}).round(4)
summary.index.name = "Revenue"
summary

,total_n,train_n,test_n,total_prop,train_prop,test_prop
Revenue,,,,,,
False,10422,8338,2084,0.8453,0.8453,0.8451
True,1908,1526,382,0.1547,0.1547,0.1549


#### Guardado de resultados

In [14]:
train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)
summary.to_csv(PROCESSED_DIR / "split_summary.csv")

for name in ["train.csv", "test.csv", "split_summary.csv"]:
    print(f"Guardado: {PROCESSED_DIR / name}")

Guardado: ..\..\Sample_Data\Processed\train.csv
Guardado: ..\..\Sample_Data\Processed\test.csv
Guardado: ..\..\Sample_Data\Processed\split_summary.csv
